# Runtime simulation for all explainers

This notebook benchmarks the explainers in `cxplain` on synthetic datasets with varying numbers of clusters, features, and observations. The goal is to approximate the runtime profile of the available explainers under controlled data sizes.

The grid includes:
- clusters: 5, 10, 20
- features: 5, 15, 30
- observations: 1000, 10000, 50000

For XKM, all flavours are benchmarked: next_best, all, within_scatter, scatter_ratio.

In [48]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [49]:
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from tqdm.auto import tqdm

from cxplain.exkmc import ExKMCExplainer
from cxplain.gradient import GradientExplainer
from cxplain.neon import NeonKMeansExplainer
from cxplain.shap import ShapExplainer
from cxplain.tree import DecisionTreeExplainer, RandomForestExplainer
from cxplain.xkm import XkmExplainer

%matplotlib inline

In [50]:
@dataclass
class RuntimeResult:
    explainer: str
    n_clusters: int
    n_features: int
    n_obs: int
    run_idx: int
    runtime_sec: float
    status: str
    global_relevance_shape: Tuple[int, ...] | None = None

CLUSTER_COUNTS = [5, 10, 15]
FEATURE_COUNTS = [5, 10, 30]
OBSERVATION_COUNTS = [100, 1000, 10000]
N_RUNS_PER_EXPLAINER = 100
SEED = 42
XKM_FLAVOURS = ['next_best', 'all', 'within_scatter', 'scatter_ratio']
EXPLAINERS = [
    *[f'xkm_{flavour}' for flavour in XKM_FLAVOURS],
    'decision_tree',
    'random_forest',
    'shap',
    'neon',
    'exkmc',
    'gradient',
]


def make_synthetic_dataset(n_clusters: int, n_features: int, n_obs: int, seed: int = SEED):
    X, labels, centers = make_blobs(
        n_samples=n_obs,
        n_features=n_features,
        centers=n_clusters,
        cluster_std=1.5,
        random_state=seed,
        return_centers=True,
    )
    return X, labels, centers


def benchmark_explainer(
    explainer_name: str,
    X: np.ndarray,
    labels: np.ndarray,
    cluster_centers: np.ndarray,
    run_idx: int,
):
    start = time.perf_counter()
    status = 'ok'
    relevance_shape = None

    try:
        if explainer_name.startswith('xkm_') or explainer_name == 'xkm':
            flavour = explainer_name.replace('xkm_', '', 1) if explainer_name.startswith('xkm_') else 'next_best'
            explainer = XkmExplainer(
                data=X,
                cluster_centers=cluster_centers,
                flavour=flavour,
                distance_metric='euclidean',
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'decision_tree':
            explainer = DecisionTreeExplainer(
                data=X,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                max_depth=5,
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'random_forest':
            explainer = RandomForestExplainer(
                data=X,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                n_estimators=25,
                max_depth=6,
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'shap':
            explainer = ShapExplainer(
                data=X,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                n_estimators=15,
                max_depth=5,
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'neon':
            explainer = NeonKMeansExplainer(
                data=X,
                cluster_centers=cluster_centers,
                predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'exkmc':
            n_clusters = cluster_centers.shape[0]
            kmeans_fitted = KMeans(n_clusters=n_clusters, random_state=SEED, n_init=10)
            kmeans_fitted.fit(X)
            explainer = ExKMCExplainer(
                data=X,
                kmeans_fitted=kmeans_fitted,
                k=n_clusters,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
                random_state=SEED,
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        elif explainer_name == 'gradient':
            explainer = GradientExplainer(
                data=X,
                cluster_centers=cluster_centers,
                cluster_predictions=labels,
                feature_names=[f'f{i}' for i in range(X.shape[1])],
            )
            result = explainer.fit().explain()
            relevance_shape = getattr(result.global_relevance, 'shape', None)

        else:
            raise ValueError(f'Unknown explainer: {explainer_name}')

    except Exception as exc:
        status = f'error: {type(exc).__name__}: {exc}'

    elapsed = time.perf_counter() - start
    return RuntimeResult(
        explainer=explainer_name,
        n_clusters=cluster_centers.shape[0],
        n_features=X.shape[1],
        n_obs=X.shape[0],
        run_idx=run_idx,
        runtime_sec=elapsed,
        status=status,
        global_relevance_shape=relevance_shape,
    )


def run_runtime_simulation(
    cluster_counts: List[int] = CLUSTER_COUNTS,
    feature_counts: List[int] = FEATURE_COUNTS,
    observation_counts: List[int] = OBSERVATION_COUNTS,
    explainers: List[str] = EXPLAINERS,
    n_runs_per_explainer: int = N_RUNS_PER_EXPLAINER,
):
    runtime_records: List[RuntimeResult] = []

    total_steps = (
        len(cluster_counts)
        * len(feature_counts)
        * len(observation_counts)
        * len(explainers)
        * n_runs_per_explainer
    )

    with tqdm(total=total_steps, desc='Runtime simulation', unit='run') as pbar:
        for n_clusters in cluster_counts:
            for n_features in feature_counts:
                for n_obs in observation_counts:
                    X, labels, cluster_centers = make_synthetic_dataset(
                        n_clusters=n_clusters,
                        n_features=n_features,
                        n_obs=n_obs,
                    )

                    for explainer_name in explainers:
                        for run_idx in range(1, n_runs_per_explainer + 1):
                            runtime_records.append(
                                benchmark_explainer(
                                    explainer_name=explainer_name,
                                    X=X,
                                    labels=labels,
                                    cluster_centers=cluster_centers,
                                    run_idx=run_idx,
                                )
                            )
                            pbar.update(1)

    results_df = pd.DataFrame([record.__dict__ for record in runtime_records])
    results_df = results_df.sort_values(
        ['n_clusters', 'n_features', 'n_obs', 'explainer', 'run_idx']
    ).reset_index(drop=True)

    # Scenario-level mean runtime per explainer (using successful runs)
    summary_df = (
        results_df.assign(runtime_sec_ok=results_df['runtime_sec'].where(results_df['status'] == 'ok'))
        .groupby(['n_clusters', 'n_features', 'n_obs', 'explainer'], as_index=False)
        .agg(
            mean_runtime_sec=('runtime_sec_ok', 'mean'),
            n_runs=('run_idx', 'count'),
            n_success=('status', lambda s: (s == 'ok').sum()),
            n_errors=('status', lambda s: (s != 'ok').sum()),
        )
        .sort_values(['n_clusters', 'n_features', 'n_obs', 'explainer'])
        .reset_index(drop=True)
    )

    return results_df, summary_df

In [51]:
# Full benchmark: each explainer is executed 100 times per scenario.
results_df, summary_df = run_runtime_simulation(n_runs_per_explainer=100)

summary_df.head(20)

Runtime simulation: 100%|█████████████████████████████████████████| 27000/27000 [38:29<00:00, 11.69run/s]


,n_clusters,n_features,n_obs,explainer,mean_runtime_sec,n_runs,n_success,n_errors
0,5,5,100,decision_tree,0.001451,100,100,0
1,5,5,100,exkmc,0.062970,100,100,0
2,5,5,100,gradient,0.001151,100,100,0
3,5,5,100,neon,0.002237,100,100,0
4,5,5,100,random_forest,0.046122,100,100,0
5,5,5,100,shap,0.033479,100,100,0
6,5,5,100,xkm_all,0.001190,100,100,0
7,5,5,100,xkm_next_best,0.001095,100,100,0
8,5,5,100,xkm_scatter_ratio,0.001117,100,100,0
9,5,5,100,xkm_within_scatter,0.001271,100,100,0


In [53]:
summary_df[summary_df['explainer'] == 'exkmc']

,n_clusters,n_features,n_obs,explainer,mean_runtime_sec,n_runs,n_success,n_errors
1,5,5,100,exkmc,0.062970,100,100,0
11,5,5,1000,exkmc,0.078973,100,100,0
21,5,5,10000,exkmc,0.062134,100,100,0
31,5,10,100,exkmc,0.027389,100,100,0
41,5,10,1000,exkmc,0.044182,100,100,0
51,5,10,10000,exkmc,0.177322,100,100,0
61,5,30,100,exkmc,0.048806,100,100,0
71,5,30,1000,exkmc,0.104029,100,100,0
81,5,30,10000,exkmc,0.802768,100,100,0
91,10,5,100,exkmc,0.062082,100,100,0


In [52]:
# Build a publication-ready runtime table for selected parameter combinations
import pandas as pd

# Ensure summary_df is available
if 'summary_df' not in globals():
    summary_df = pd.read_csv('runtime_simulation_results_summary.csv')

requested = pd.DataFrame([
    {'n_clusters': 5, 'n_features': 5, 'n_obs': 100},
    {'n_clusters': 10, 'n_features': 10, 'n_obs': 1000},
    {'n_clusters': 15, 'n_features': 30, 'n_obs': 10000},
])

subset = summary_df.merge(requested, on=['n_clusters', 'n_features', 'n_obs'], how='inner').copy()

# Keep successful mean runtime and success/error counts for transparency
subset = subset[[
    'n_clusters', 'n_features', 'n_obs', 'explainer',
    'mean_runtime_sec', 'n_success', 'n_errors'
]].sort_values(['n_clusters', 'n_features', 'n_obs', 'explainer'])

# Wide table for paper readability
table_df = subset.pivot_table(
    index='explainer',
    columns=['n_clusters', 'n_features', 'n_obs'],
    values='mean_runtime_sec',
    aggfunc='first'
)

# Rename multiindex columns to compact scenario names
table_df.columns = [f"k={k}, p={p}, n={n}" for (k, p, n) in table_df.columns]

# Optional sorting: XKM first, then the others
xkm_rows = [r for r in table_df.index if str(r).startswith('xkm_')]
other_rows = [r for r in table_df.index if r not in xkm_rows]
table_df = table_df.loc[xkm_rows + sorted(other_rows)]

print('Selected runtime table (seconds):')
display(table_df)

caption = (
    'Mean runtime (seconds) of each explainer for three representative synthetic-data scenarios. '
    'Each reported value corresponds to the average runtime over the configured repetitions for that scenario; '
    'in this run, one repetition per explainer was executed (n\\_runs=1). '
    'Scenarios vary jointly in cluster count (k), feature dimensionality (p), and sample size (n).'
)
label = 'tab:runtime_selected_scenarios'

latex_table = table_df.to_latex(
    float_format=lambda x: f"{x:.4f}",
    na_rep='--',
    caption=caption,
    label=label,
    escape=False,
)

print('\nLaTeX table:\n')
print(latex_table)

# Also show run-quality checks
quality = subset.groupby(['n_clusters', 'n_features', 'n_obs', 'explainer'], as_index=False)[['n_success', 'n_errors']].first()
print('\nRun status checks (success/error counts):')
display(quality.sort_values(['n_clusters','n_features','n_obs','explainer']))

Selected runtime table (seconds):


,"k=5, p=5, n=100","k=10, p=10, n=1000","k=15, p=30, n=10000"
explainer,,,
xkm_all,0.001190,0.001285,0.037548
xkm_next_best,0.001095,0.001468,0.044553
xkm_scatter_ratio,0.001117,0.001728,0.053864
xkm_within_scatter,0.001271,0.000873,0.008625
decision_tree,0.001451,0.006982,0.182525
exkmc,0.062970,0.105546,1.724140
gradient,0.001151,0.001556,0.007288
neon,0.002237,0.004221,0.057071
random_forest,0.046122,0.076118,0.853946



LaTeX table:

\begin{table}
\caption{Mean runtime (seconds) of each explainer for three representative synthetic-data scenarios. Each reported value corresponds to the average runtime over the configured repetitions for that scenario; in this run, one repetition per explainer was executed (n\_runs=1). Scenarios vary jointly in cluster count (k), feature dimensionality (p), and sample size (n).}
\label{tab:runtime_selected_scenarios}
\begin{tabular}{lrrr}
\toprule
 & k=5, p=5, n=100 & k=10, p=10, n=1000 & k=15, p=30, n=10000 \\
explainer &  &  &  \\
\midrule
xkm_all & 0.0012 & 0.0013 & 0.0375 \\
xkm_next_best & 0.0011 & 0.0015 & 0.0446 \\
xkm_scatter_ratio & 0.0011 & 0.0017 & 0.0539 \\
xkm_within_scatter & 0.0013 & 0.0009 & 0.0086 \\
decision_tree & 0.0015 & 0.0070 & 0.1825 \\
exkmc & 0.0630 & 0.1055 & 1.7241 \\
gradient & 0.0012 & 0.0016 & 0.0073 \\
neon & 0.0022 & 0.0042 & 0.0571 \\
random_forest & 0.0461 & 0.0761 & 0.8539 \\
shap & 0.0335 & 0.0869 & 1.2386 \\
\bottomrule
\end{tabula

,n_clusters,n_features,n_obs,explainer,n_success,n_errors
0,5,5,100,decision_tree,100,0
1,5,5,100,exkmc,100,0
2,5,5,100,gradient,100,0
3,5,5,100,neon,100,0
4,5,5,100,random_forest,100,0
5,5,5,100,shap,100,0
6,5,5,100,xkm_all,100,0
7,5,5,100,xkm_next_best,100,0
8,5,5,100,xkm_scatter_ratio,100,0
9,5,5,100,xkm_within_scatter,100,0


In [54]:
# Flipped full runtime table: rows = scenarios, columns = explainers (all scenarios)
import pandas as pd

# Ensure summary_df is available
if 'summary_df' not in globals():
    summary_df = pd.read_csv('runtime_simulation_results_summary.csv')

# Keep only the fields needed
full_subset = summary_df[[
    'n_clusters', 'n_features', 'n_obs', 'explainer', 'mean_runtime_sec', 'n_runs', 'n_success', 'n_errors'
]].copy()

# Build wide table with all scenarios, then flip relative to prior layout
# Prior layout had rows=explainer and columns=scenario; this creates rows=scenario and cols=explainer
full_table = full_subset.pivot_table(
    index=['n_clusters', 'n_features', 'n_obs'],
    columns='explainer',
    values='mean_runtime_sec',
    aggfunc='first'
).sort_index()

# Optional: put XKM columns first
xkm_cols = [c for c in full_table.columns if str(c).startswith('xkm_')]
other_cols = [c for c in full_table.columns if c not in xkm_cols]
full_table = full_table[xkm_cols + sorted(other_cols)]

# Compact scenario index labels
full_table.index = [f"k={k}, p={p}, n={n}" for (k, p, n) in full_table.index]
full_table.index.name = 'scenario'

print('Full runtime table (seconds), flipped:')
display(full_table)

# Use observed n_runs from summary if constant; otherwise indicate mixed runs
unique_runs = sorted(full_subset['n_runs'].dropna().unique().tolist())
if len(unique_runs) == 1:
    runs_text = f"n\\_runs={int(unique_runs[0])}"
else:
    runs_text = 'mixed n\\_runs across scenarios'

caption = (
    'Mean runtime (seconds) of each explainer across all synthetic-data scenarios. '
    'Rows correspond to scenarios (k, p, n) and columns to explainers. '
    f'Values are scenario-level means over successful runs ({runs_text}).'
)
label = 'tab:runtime_all_scenarios_flipped'

latex_full_table = full_table.to_latex(
    float_format=lambda x: f"{x:.4f}",
    na_rep='--',
    caption=caption,
    label=label,
    escape=False,
)

print('\nLaTeX table (all scenarios, flipped):\n')
print(latex_full_table)

# Scenario-level run-quality summary (aggregated across explainers)
quality_all = full_subset.groupby(['n_clusters', 'n_features', 'n_obs'], as_index=False)[['n_success', 'n_errors']].sum()
print('\nScenario-level run status totals (across explainers):')
display(quality_all.sort_values(['n_clusters', 'n_features', 'n_obs']))

Full runtime table (seconds), flipped:


explainer,xkm_all,xkm_next_best,xkm_scatter_ratio,xkm_within_scatter,decision_tree,exkmc,gradient,neon,random_forest,shap
scenario,,,,,,,,,,
"k=5, p=5, n=100",0.001190,0.001095,0.001117,0.001271,0.001451,0.062970,0.001151,0.002237,0.046122,0.033479
"k=5, p=5, n=1000",0.001917,0.001816,0.001918,0.001159,0.003191,0.078973,0.001567,0.004716,0.062733,0.069199
"k=5, p=5, n=10000",0.005546,0.007292,0.007137,0.002400,0.034367,0.062134,0.000780,0.016300,0.375862,0.695780
"k=5, p=10, n=100",0.000509,0.000544,0.000524,0.000528,0.000698,0.027389,0.000488,0.000757,0.014857,0.010018
"k=5, p=10, n=1000",0.000746,0.000884,0.000807,0.000616,0.001935,0.044182,0.000546,0.001551,0.021982,0.019682
"k=5, p=10, n=10000",0.004784,0.005538,0.006222,0.001439,0.020901,0.177322,0.002209,0.018637,0.137118,0.203949
"k=5, p=30, n=100",0.001032,0.001092,0.001206,0.001310,0.001921,0.048806,0.001246,0.001734,0.037819,0.029640
"k=5, p=30, n=1000",0.002089,0.002610,0.002427,0.001119,0.008805,0.104029,0.001210,0.005047,0.068253,0.050180
"k=5, p=30, n=10000",0.019568,0.021920,0.025912,0.009065,0.095133,0.802768,0.008153,0.047481,0.391905,0.372984



LaTeX table (all scenarios, flipped):

\begin{table}
\caption{Mean runtime (seconds) of each explainer across all synthetic-data scenarios. Rows correspond to scenarios (k, p, n) and columns to explainers. Values are scenario-level means over successful runs (n\_runs=100).}
\label{tab:runtime_all_scenarios_flipped}
\begin{tabular}{lrrrrrrrrrr}
\toprule
explainer & xkm_all & xkm_next_best & xkm_scatter_ratio & xkm_within_scatter & decision_tree & exkmc & gradient & neon & random_forest & shap \\
scenario &  &  &  &  &  &  &  &  &  &  \\
\midrule
k=5, p=5, n=100 & 0.0012 & 0.0011 & 0.0011 & 0.0013 & 0.0015 & 0.0630 & 0.0012 & 0.0022 & 0.0461 & 0.0335 \\
k=5, p=5, n=1000 & 0.0019 & 0.0018 & 0.0019 & 0.0012 & 0.0032 & 0.0790 & 0.0016 & 0.0047 & 0.0627 & 0.0692 \\
k=5, p=5, n=10000 & 0.0055 & 0.0073 & 0.0071 & 0.0024 & 0.0344 & 0.0621 & 0.0008 & 0.0163 & 0.3759 & 0.6958 \\
k=5, p=10, n=100 & 0.0005 & 0.0005 & 0.0005 & 0.0005 & 0.0007 & 0.0274 & 0.0005 & 0.0008 & 0.0149 & 0.0100 \\
k=5, p=1

,n_clusters,n_features,n_obs,n_success,n_errors
0,5,5,100,1000,0
1,5,5,1000,1000,0
2,5,5,10000,1000,0
3,5,10,100,1000,0
4,5,10,1000,1000,0
5,5,10,10000,1000,0
6,5,30,100,1000,0
7,5,30,1000,1000,0
8,5,30,10000,1000,0
9,10,5,100,1000,0


In [19]:
results_df.to_csv('runtime_simulation_results_raw.csv', index=False)
summary_df.to_csv('runtime_simulation_results_summary.csv', index=False)